<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "028"
iso_country = "Antigua and Barbuda"
year = 2024

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    "TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Antigua and Barbuda': 

Number of trade records present in dataset: 13161
Number of partner countries present in dataset: 131
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$581.4m
Total weight within the dataset: 426.8 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,028,Antigua and Barbuda,340,Honduras,Import,Sea,440710,"440710 | Wood, coniferous; sa",4407,4407 | Wood sawn or chipped,44,44 | Wood and articles of,311,"Wood, sawn or chipped lengthwise, sliced or pe...",Wood and Paper,30,LUM - Wood products,244101.189,NaN,244101.189,m³,34570.716,187000.00,477510.745922,2540.0,1578.282340,753646.777449,['all'],['all'],"1 - Reporter, Partner and HS4 code",8.72,1,19578.39,244101.19,224522.80,3. Primary Int. Inv. without Vess Type,4180.127832,4172.221539,4173.124077,4173.124077,4176.607912,4177.460511,4332.188882,7117.484125,4625.401385,4348.409581,4360.250448,4502.015150,4256.331052,6845.558495,4868.581293,4528.672624,4453.055325,4613.536201,4190.772907,7536.895708,5738.914081,5273.729921,4965.898514,4771.784191,4126.376357,8236.363242,6678.034823,6321.223681,5170.416758,5056.921674,4063.473934,7993.920032,7650.828709,7947.794319,5542.751165,5266.855702,3999.314135,7803.283467,7444.033456,7959.292505,6129.017924,5343.467755,3938.913532,7007.163191,7308.619412,7902.118446,6158.303042,5392.337386,3877.463361,6598.375704,6714.636898,7900.363769,6476.325848,5447.605929,311
1,1,028,Antigua and Barbuda,328,Guyana,Import,Sea,440729,"440729 | Wood, tropical; (as",4407,4407 | Wood sawn or chipped,44,44 | Wood and articles of,311,"Wood, sawn or chipped lengthwise, sliced or pe...",Wood and Paper,30,LUM - Wood products,167403.233,NaN,167403.233,m³,457.401,191551.04,327474.204395,1355.0,841.957705,275719.429579,['Container'],['Container'],"1 - Reporter, Partner and HS4 code",7.85,1,12184.66,167403.23,155218.57,1. Primary Int. Inv. with Vess Type,1441.138358,1438.622096,1438.867999,1438.867999,1439.995579,1439.995579,1507.504005,2671.770061,1662.825235,1543.778500,1547.971054,1586.741298,1476.111195,2616.589645,1750.228417,1603.183080,1566.393198,1637.335226,1448.578156,2845.363884,2170.367232,1898.126616,1756.126414,1706.588226,1421.555211,3025.319097,2489.490166,2332.863562,1880.159299,1829.035519,1395.042352,3065.737127,2929.040536,2954.224659,2038.288024,1921.342848,1367.998037,3002.724142,2877.529690,3152.756581,2332.072318,1956.558107,1342.520360,2678.502795,2791.712887,2925.423786,2192.990144,1979.675844,1316.541233,2460.433739,2501.095261,3119.793546,2577.387225,1992.785392,311


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [14]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $177.5m (30.5%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
02 | Meat and edible meat  29855206.1  12488981.7      5.1      2.9
22 | Beverages, spirits a  29484295.9  79757059.2      5.1     18.7
21 | Miscellaneous edible  18096683.5   5135860.8      3.1      1.2
04 | Dairy produce; birds  14336813.2   7660035.2      2.5      1.8
07 | Vegetables and certa  12721863.9  17092054.5      2.2      4.0
19 | Preparations of cere  11926328.2   4613417.7      2.1      1.1
08 | Fruit and nuts, edib   9373893.6   8936140.4      1.6      2.1
20 | Preparations of vege   8961350.5   6405066.7      1.5      1.5
16 | Meat, fish, crustace   8636378.9   1909815.3      1.5      0.4
03 | Fish and crustaceans   8337647.4   1517152.6      1.4      0.4
23 | Food industries, res   5634159.6   4180510.8      1.0      1.0
11 | Products o

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [15]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $181.0m (31.1%), ... 

                    cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                  
Food & Beverages  89892628.5  110569300.0     15.5     25.9
Agriculture       82805038.0   67456583.0     14.2     15.8
Fishing            8337647.4    1517152.6      1.4      0.4



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $150.5m (25.9%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     63611861.6  35864013.3     10.9      8.4
OMT - Meat products n.e.c.     28929676.8  12274254.8      5.0      2.9
V_F - Vegetables, fruit, nuts  20723459.7  25059241.3      3.6      5.9
MIL - Dairy products           15193185.0   7862863.4      2.6      1.8
CMT - Bovine meat prods         7577658.2   1739915.7      1.3      0.4
VOL - Vegetable oils and fats   4423099.4   3221573.7      0.8      0.8
OCR - Crops n.e.c.              3626307.6   1254314.5      0.6      0.3
SGR - Sugar                     2523244.6   7591503.9      0.4      1.8
PDR - Paddy rice                1970088.5   3422043.7      0.3      0.8
FSH - Fishing                    851546.6    185435.2      0.1      0.0
OAP - Animal products n.e.c.     557083.2    236872.3      0.1      0.

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 30.5% 
GTAP 25.9% 
EORA 31.1%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_21416/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,028,Antigua and Barbuda,340,Honduras,Import,Sea,440710,"440710 | Wood, coniferous; sa",4407,4407 | Wood sawn or chipped,44,44 | Wood and articles of,311,"Wood, sawn or chipped lengthwise, sliced or pe...",Wood and Paper,30,LUM - Wood products,244101.189,NaN,244101.189,m³,34570.716,187000.00,477510.745922,2540.0,1578.282340,753646.777449,['all'],['all'],"1 - Reporter, Partner and HS4 code",8.72,1,19578.39,244101.19,224522.80,3. Primary Int. Inv. without Vess Type,4180.127832,4172.221539,4173.124077,4173.124077,4176.607912,4177.460511,4332.188882,7117.484125,4625.401385,4348.409581,4360.250448,4502.015150,4256.331052,6845.558495,4868.581293,4528.672624,4453.055325,4613.536201,4190.772907,7536.895708,5738.914081,5273.729921,4965.898514,4771.784191,4126.376357,8236.363242,6678.034823,6321.223681,5170.416758,5056.921674,4063.473934,7993.920032,7650.828709,7947.794319,5542.751165,5266.855702,3999.314135,7803.283467,7444.033456,7959.292505,6129.017924,5343.467755,3938.913532,7007.163191,7308.619412,7902.118446,6158.303042,5392.337386,3877.463361,6598.375704,6714.636898,7900.363769,6476.325848,5447.605929,311,NaN
1,1,028,Antigua and Barbuda,328,Guyana,Import,Sea,440729,"440729 | Wood, tropical; (as",4407,4407 | Wood sawn or chipped,44,44 | Wood and articles of,311,"Wood, sawn or chipped lengthwise, sliced or pe...",Wood and Paper,30,LUM - Wood products,167403.233,NaN,167403.233,m³,457.401,191551.04,327474.204395,1355.0,841.957705,275719.429579,['Container'],['Container'],"1 - Reporter, Partner and HS4 code",7.85,1,12184.66,167403.23,155218.57,1. Primary Int. Inv. with Vess Type,1441.138358,1438.622096,1438.867999,1438.867999,1439.995579,1439.995579,1507.504005,2671.770061,1662.825235,1543.778500,1547.971054,1586.741298,1476.111195,2616.589645,1750.228417,1603.183080,1566.393198,1637.335226,1448.578156,2845.363884,2170.367232,1898.126616,1756.126414,1706.588226,1421.555211,3025.319097,2489.490166,2332.863562,1880.159299,1829.035519,1395.042352,3065.737127,2929.040536,2954.224659,2038.288024,1921.342848,1367.998037,3002.724142,2877.529690,3152.756581,2332.072318,1956.558107,1342.520360,2678.502795,2791.712887,2925.423786,2192.990144,1979.675844,1316.541233,2460.433739,2501.095261,3119.793546,2577.387225,1992.785392,311,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $103.8m of $581.4m (17.9%)
Volume: 81.9 of 426.8 million kilograms (19.2%)
Value-to-Weight: 1.27 from 1.36 US$ per kg (+-6.9%)
Average distance to market (volume-weighted): 2126.9 from 2096.3 nm (+1.5%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    444  34436363.3  23547844.5      5.9      5.5
1        Meat and Prep     66  21814938.3  10556744.9      3.8      2.5
2           Vegetables    377  16405150.4  20110039.0      2.8      4.7
3                Dairy    122  13807092.6   7420601.6      2.4      1.7
4               Fruits    208   9105243.8   8810166.7      1.6      2.1
5        Fats and Oils     90   4057564.0   2754455.9      0.7      0.6
6      Sugar and Honey     72   4029770.0   8098457.5      0.7      1.9
7  Unprocessed Cereals     22    154213.5    580028.6      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [19]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,13825681.9,24093996.7,15138457.1,14094045.8,14132364.7,14510156.1,12833924.9,27543426.0,26316616.7,26707584.5,18461322.7,17436393.7,12142546.6,22224087.4,22593156.9,27831100.4,22975372.4,18073702.3
1,Fats and Oils,4865735.4,8357222.8,5309083.5,4945369.3,4958550.1,5096951.4,4528258.3,9523410.6,9146760.1,9294475.4,6449427.0,6091964.9,4292734.5,7741475.8,7870578.3,9614279.1,7987368.8,6304461.8
2,Fruits,7521016.4,13242805.6,8272223.7,7688366.0,7709250.5,7907375.5,6968373.8,15168745.3,14494504.1,14654884.9,10120403.6,9546267.1,6582855.8,12202123.8,12404893.1,15399930.3,12714757.4,9898888.2
3,Meat and Prep,16218031.5,28725635.5,17884144.0,16605483.8,16650581.5,17068643.6,15009876.7,32955889.4,31486859.9,31764929.9,21918314.0,20662105.7,14166603.1,26454839.0,26892265.2,33529532.9,27698495.3,21429918.9
4,Processed Cereals,17856983.4,31137413.8,19558284.0,18207393.6,18257098.1,18743548.5,16574176.9,35603820.3,34027193.2,34522053.7,23875774.7,22529138.1,15680025.2,28734726.5,29219230.7,36009540.6,29716997.1,23350876.1
5,Sugar and Honey,7821089.8,12918479.5,8399239.1,7870080.4,7890970.7,8140821.6,7328507.5,14564346.6,14021855.2,14448851.3,10086021.1,9563046.0,6985890.1,12019856.4,12226131.8,14496841.0,12029934.9,9879379.8
6,Unprocessed Cereals,674400.1,1162746.3,735046.9,685562.9,687436.1,706521.3,627235.5,1325531.9,1267149.3,1291100.0,894364.0,844859.1,594398.0,1074165.9,1092463.8,1334956.1,1100521.6,875305.9
7,Vegetables,24928556.1,43800025.9,27389247.8,25467221.3,25536627.3,26198482.7,23106693.1,50180193.8,47942126.6,48509032.1,33494524.0,31599794.2,21833523.0,40376361.3,41053752.1,50859511.8,41986115.3,32762691.6


### 3.3.2 Total FEM costs by scenario (% cif)

In [18]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.74,0.10,0.02,0.02,0.05,1.07,0.98,1.00,0.41,0.33,0.73,0.76,1.14,0.78,0.43
1,Fats and Oils,0.86,0.11,0.02,0.02,0.06,1.23,1.14,1.17,0.47,0.39,0.85,0.88,1.31,0.91,0.50
2,Fruits,0.63,0.08,0.02,0.02,0.04,0.90,0.83,0.84,0.35,0.28,0.62,0.64,0.97,0.67,0.36
3,Meat and Prep,0.57,0.08,0.02,0.02,0.04,0.82,0.76,0.77,0.32,0.26,0.56,0.58,0.89,0.62,0.33
4,Processed Cereals,0.39,0.05,0.01,0.01,0.03,0.55,0.51,0.52,0.21,0.17,0.38,0.39,0.59,0.41,0.22
5,Sugar and Honey,1.26,0.14,0.01,0.02,0.08,1.80,1.66,1.77,0.68,0.55,1.25,1.30,1.86,1.25,0.72
6,Unprocessed Cereals,3.17,0.39,0.07,0.08,0.21,4.53,4.15,4.30,1.73,1.41,3.11,3.23,4.80,3.28,1.82
7,Vegetables,1.15,0.15,0.03,0.04,0.08,1.65,1.51,1.55,0.63,0.52,1.13,1.17,1.77,1.23,0.67
